# Notebook 1: Data Collection and Preprocessing
**DS340W Capstone — Group 60**

This notebook downloads NYC crime data, weather data, and holiday information, then cleans, merges, and prepares everything for modeling.

In [ ]:
# Cell 1: Setup
from google.colab import drive
drive.mount('/content/drive')

import os
base_path = '/content/drive/MyDrive/capstone_project'
for folder in ['raw_data', 'processed_data', 'notebooks', 'outputs']:
    os.makedirs(f'{base_path}/{folder}', exist_ok=True)

!pip install sodapy openmeteo-requests requests-cache retry-requests geopandas shapely -q

print("Setup complete!")

In [ ]:
# Cell 2: Download NYC Crime Data (2015-2020)
import pandas as pd
from sodapy import Socrata

print("Downloading NYC crime data...")
client = Socrata("data.cityofnewyork.us", None)
all_data = []

for year in [2015, 2016, 2017, 2018, 2019, 2020]:
    print(f"  Downloading {year}...")
    results = client.get(
        "qgea-i56i",
        where=f"cmplnt_fr_dt >= '{year}-01-01T00:00:00' AND cmplnt_fr_dt < '{year+1}-01-01T00:00:00'",
        limit=500000,
        select="cmplnt_fr_dt, addr_pct_cd, ofns_desc, law_cat_cd, latitude, longitude"
    )
    all_data.append(pd.DataFrame.from_records(results))
    print(f"    Got {len(results)} records")

crime_raw = pd.concat(all_data, ignore_index=True)
print(f"Total records: {len(crime_raw)}")
crime_raw.to_csv(f'{base_path}/raw_data/nyc_crime_raw.csv', index=False)
print("Saved!")

In [ ]:
# Cell 3: Filter to 4 crime types
crime_raw = pd.read_csv(f'{base_path}/raw_data/nyc_crime_raw.csv')

crime_type_mapping = {
    'FELONY ASSAULT': 'ASSAULT',
    'ASSAULT 3 & RELATED OFFENSES': 'ASSAULT',
    'ROBBERY': 'ROBBERY',
    'CRIMINAL MISCHIEF & RELATED OF': 'CRIMINAL_DAMAGE',
    'PETIT LARCENY': 'THEFT',
    'GRAND LARCENY': 'THEFT',
    'GRAND LARCENY OF MOTOR VEHICLE': 'THEFT',
}

crime_raw['crime_type'] = crime_raw['ofns_desc'].map(crime_type_mapping)
crime_filtered = crime_raw.dropna(subset=['crime_type']).copy()

print(f"Records after filtering: {len(crime_filtered)}")
print(crime_filtered['crime_type'].value_counts())

In [ ]:
# Cell 4: Clean dates, precincts, coordinates
crime_filtered['date'] = pd.to_datetime(crime_filtered['cmplnt_fr_dt'], errors='coerce')
crime_filtered = crime_filtered.dropna(subset=['date'])
crime_filtered['date'] = pd.to_datetime(crime_filtered['date'].dt.date)

crime_filtered['precinct'] = pd.to_numeric(crime_filtered['addr_pct_cd'], errors='coerce')
crime_filtered = crime_filtered.dropna(subset=['precinct'])
crime_filtered['precinct'] = crime_filtered['precinct'].astype(int)
crime_filtered = crime_filtered[(crime_filtered['precinct'] >= 1) & (crime_filtered['precinct'] <= 77)]

crime_filtered['latitude'] = pd.to_numeric(crime_filtered['latitude'], errors='coerce')
crime_filtered['longitude'] = pd.to_numeric(crime_filtered['longitude'], errors='coerce')
crime_filtered = crime_filtered.dropna(subset=['latitude', 'longitude'])

crime_filtered = crime_filtered[
    (crime_filtered['date'] >= '2015-01-01') & (crime_filtered['date'] <= '2020-03-10')
]

print(f"Clean dataset: {len(crime_filtered)} records")
print(f"Precincts: {crime_filtered['precinct'].nunique()}")

In [ ]:
# Cell 5: Aggregate daily crime counts per precinct
from itertools import product
import numpy as np

daily_counts = crime_filtered.groupby(['date', 'precinct', 'crime_type']).size().reset_index(name='count')
daily_pivot = daily_counts.pivot_table(index=['date', 'precinct'], columns='crime_type', values='count', fill_value=0).reset_index()
daily_pivot.columns = ['date', 'precinct', 'assault', 'criminal_damage', 'robbery', 'theft']

all_dates = pd.date_range(start='2015-01-01', end='2020-03-10', freq='D')
all_precincts = sorted(crime_filtered['precinct'].unique())
complete_index = pd.DataFrame(list(product(all_dates, all_precincts)), columns=['date', 'precinct'])

daily_complete = complete_index.merge(daily_pivot, on=['date', 'precinct'], how='left').fillna(0)
for col in ['assault', 'criminal_damage', 'robbery', 'theft']:
    daily_complete[col] = daily_complete[col].astype(int)

print(f"Complete dataset: {len(daily_complete)} rows ({len(all_dates)} days x {len(all_precincts)} precincts)")
daily_complete.to_csv(f'{base_path}/processed_data/daily_crime_counts.csv', index=False)
print("Saved!")

In [ ]:
# Cell 6: Download weather data
import openmeteo_requests
import requests_cache
from retry_requests import retry

cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

url = "https://archive-api.open-meteo.com/v1/archive"
weather_frames = []

for year in range(2015, 2021):
    end_date = f"{year}-12-31" if year < 2020 else "2020-03-10"
    print(f"  Downloading weather {year}...")
    params = {
        "latitude": 40.7831, "longitude": -73.9712,
        "start_date": f"{year}-01-01", "end_date": end_date,
        "daily": ["temperature_2m_mean", "relative_humidity_2m_mean", "wind_speed_10m_max", "precipitation_sum"],
        "timezone": "America/New_York"
    }
    responses = openmeteo.weather_api(url, params=params)
    daily = responses[0].Daily()
    start_date = pd.to_datetime(daily.Time(), unit="s", utc=True)
    end_date_ts = pd.to_datetime(daily.TimeEnd(), unit="s", utc=True)
    dates = pd.date_range(start=start_date, end=end_date_ts, freq="D", inclusive="left")
    weather_frames.append(pd.DataFrame({
        "date": dates,
        "temperature": daily.Variables(0).ValuesAsNumpy()[:len(dates)],
        "humidity": daily.Variables(1).ValuesAsNumpy()[:len(dates)],
        "wind_speed": daily.Variables(2).ValuesAsNumpy()[:len(dates)],
        "precipitation": daily.Variables(3).ValuesAsNumpy()[:len(dates)],
    }))

weather = pd.concat(weather_frames, ignore_index=True)
weather['date'] = weather['date'].dt.tz_localize(None)
weather['date'] = pd.to_datetime(weather['date'].dt.date)
print(f"Weather data: {len(weather)} days")
weather.to_csv(f'{base_path}/raw_data/nyc_weather.csv', index=False)
print("Saved!")

In [ ]:
# Cell 7: Create holiday and temporal features
us_holidays = pd.to_datetime([
    '2015-01-01','2015-01-19','2015-02-16','2015-05-25','2015-07-03','2015-09-07','2015-10-12','2015-11-11','2015-11-26','2015-12-25',
    '2016-01-01','2016-01-18','2016-02-15','2016-05-30','2016-07-04','2016-09-05','2016-10-10','2016-11-11','2016-11-24','2016-12-26',
    '2017-01-02','2017-01-16','2017-02-20','2017-05-29','2017-07-04','2017-09-04','2017-10-09','2017-11-10','2017-11-23','2017-12-25',
    '2018-01-01','2018-01-15','2018-02-19','2018-05-28','2018-07-04','2018-09-03','2018-10-08','2018-11-12','2018-11-22','2018-12-25',
    '2019-01-01','2019-01-21','2019-02-18','2019-05-27','2019-07-04','2019-09-02','2019-10-14','2019-11-11','2019-11-28','2019-12-25',
    '2020-01-01','2020-01-20','2020-02-17',
])

all_dates = pd.date_range(start='2015-01-01', end='2020-03-10', freq='D')
temporal = pd.DataFrame({'date': all_dates})
temporal['weekend'] = (temporal['date'].dt.dayofweek >= 5).astype(int)
temporal['holiday'] = temporal['date'].isin(us_holidays).astype(int)
temporal['day_of_week'] = temporal['date'].dt.dayofweek
temporal['month'] = temporal['date'].dt.month

temporal.to_csv(f'{base_path}/processed_data/temporal_features.csv', index=False)
print(f"Temporal features: {len(temporal)} days, {temporal['holiday'].sum()} holidays")

In [ ]:
# Cell 8: Compute derived weather features (Appendix A of parent paper)
weather = pd.read_csv(f'{base_path}/raw_data/nyc_weather.csv')
weather['date'] = pd.to_datetime(weather['date'])
weather = weather.ffill().bfill()

T = weather['temperature'].values
RH = weather['humidity'].values
V = weather['wind_speed'].values

e = (RH / 100) * 6.105 * np.exp((17.2 * T) / (237.7 + T))
AT = 1.07 * T + 0.2 * e - 0.65 * V - 2.7

a, b, c, d_coef, e_coef, f_coef = 0.151977, 8.313659, 1.676311, 0.00391838, 0.023101, 4.68035
Tw = AT * np.arctan(a * np.sqrt(RH + b)) + np.arctan(AT + RH) - np.arctan(RH - c) + d_coef * (RH ** 1.5) * np.arctan(e_coef * RH) - f_coef
DI = 0.5 * Tw + 0.5 * AT

weather['water_vapor_pressure'] = e
weather['apparent_temp'] = AT
weather['wet_bulb_temp'] = Tw
weather['discomfort_index'] = DI

weather.to_csv(f'{base_path}/processed_data/weather_features.csv', index=False)
print("Weather features saved!")

In [ ]:
# Cell 9: Merge all data
daily_crime = pd.read_csv(f'{base_path}/processed_data/daily_crime_counts.csv')
weather_feat = pd.read_csv(f'{base_path}/processed_data/weather_features.csv')
temporal_feat = pd.read_csv(f'{base_path}/processed_data/temporal_features.csv')

for df in [daily_crime, weather_feat, temporal_feat]:
    df['date'] = pd.to_datetime(df['date'])

merged = daily_crime.merge(temporal_feat, on='date', how='left').merge(weather_feat, on='date', how='left')
merged = merged.sort_values(['precinct', 'date']).reset_index(drop=True)

for crime_col in ['assault', 'criminal_damage', 'robbery', 'theft']:
    merged[f'{crime_col}_weekday_avg'] = merged.groupby('precinct')[crime_col].transform(lambda x: x.rolling(30, min_periods=1).mean())
    merged[f'{crime_col}_month_avg'] = merged.groupby('precinct')[crime_col].transform(lambda x: x.rolling(30, min_periods=1).mean())
    merged[f'{crime_col}_prev'] = merged.groupby('precinct')[crime_col].shift(1).fillna(0)

merged.to_csv(f'{base_path}/processed_data/final_dataset.csv', index=False)
print(f"Final dataset: {merged.shape}")

In [ ]:
# Cell 10: Build precinct adjacency matrix
import geopandas as gpd
import requests, json

print("Downloading precinct boundaries...")
url = "https://services5.arcgis.com/GfwWNkhOj9bNBqoJ/arcgis/rest/services/NYC_Police_Precincts/FeatureServer/0/query?where=1%3D1&outFields=*&outSR=4326&f=geojson"
response = requests.get(url)
data = response.json()

with open(f'{base_path}/raw_data/nyc_precincts.geojson', 'w') as f:
    json.dump(data, f)

precincts_gdf = gpd.read_file(f'{base_path}/raw_data/nyc_precincts.geojson')
precinct_col = [c for c in precincts_gdf.columns if 'precinct' in c.lower()][0]
precincts_gdf['precinct_id'] = pd.to_numeric(precincts_gdf[precinct_col], errors='coerce').astype(int)

crime_precincts = sorted(pd.read_csv(f'{base_path}/processed_data/daily_crime_counts.csv')['precinct'].unique())
precincts_gdf = precincts_gdf[precincts_gdf['precinct_id'].isin(crime_precincts)].sort_values('precinct_id').reset_index(drop=True)

n = len(precincts_gdf)
precinct_ids = precincts_gdf['precinct_id'].values
adj_matrix = np.zeros((n, n), dtype=int)

for i in range(n):
    for j in range(i+1, n):
        if precincts_gdf.iloc[i].geometry.buffer(0.001).intersects(precincts_gdf.iloc[j].geometry):
            adj_matrix[i, j] = 1
            adj_matrix[j, i] = 1
np.fill_diagonal(adj_matrix, 1)

np.save(f'{base_path}/processed_data/adjacency_matrix.npy', adj_matrix)
np.save(f'{base_path}/processed_data/precinct_ids.npy', precinct_ids)
print(f"Adjacency matrix: {adj_matrix.shape}, Edges: {(adj_matrix.sum() - n) // 2}")

In [ ]:
# Cell 11: Train/test split
merged = pd.read_csv(f'{base_path}/processed_data/final_dataset.csv')
merged['date'] = pd.to_datetime(merged['date'])

train_data = merged[merged['date'] <= '2019-12-31'].copy()
test_data = merged[(merged['date'] >= '2020-01-01') & (merged['date'] <= '2020-01-07')].copy()

train_data.to_csv(f'{base_path}/processed_data/train_data.csv', index=False)
test_data.to_csv(f'{base_path}/processed_data/test_data.csv', index=False)
print(f"Train: {len(train_data)} rows, Test: {len(test_data)} rows")